# 🌿 Plant Disease Detector — Training Notebook
**Model**: EfficientNet-B3 (Transfer Learning)  
**Dataset**: New Plant Diseases Dataset (Kaggle)  
**Target**: 38-class plant disease classification  

### Steps
1. Mount Google Drive & set up Kaggle
2. Download dataset
3. Build data pipeline
4. Define model (EfficientNet-B3)
5. Train with best practices
6. Evaluate & visualize results
7. Export model for deployment

## Step 1 — Setup & Mount Google Drive

In [ ]:
# Mount Google Drive (your files will persist here)
from google.colab import drive
drive.mount('/content/drive')

import os

# Create project folder structure in your Drive
BASE_DIR = '/content/drive/MyDrive/PlantDiseaseDetector'
os.makedirs(f'{BASE_DIR}/models', exist_ok=True)
os.makedirs(f'{BASE_DIR}/logs', exist_ok=True)
os.makedirs(f'{BASE_DIR}/data', exist_ok=True)

print('✅ Google Drive mounted and folders created!')
print(f'📁 Project folder: {BASE_DIR}')

## Step 2 — Install Dependencies & Kaggle API

In [ ]:
# Install required libraries
!pip install kaggle timm grad-cam -q

print('✅ Libraries installed!')

In [ ]:
# Upload your kaggle.json API key
# Go to kaggle.com -> Settings -> API -> Create New Token -> download kaggle.json
# Then upload it here:

from google.colab import files
print('📤 Please upload your kaggle.json file...')
uploaded = files.upload()

# Move it to the right location
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print('✅ Kaggle API key configured!')

## Step 3 — Download Dataset

In [ ]:
# Download the dataset from Kaggle (~3GB, will take a few minutes)
# We download to /content (fast SSD) not Drive (slow) for training speed

DATA_DIR = '/content/plant-diseases'
os.makedirs(DATA_DIR, exist_ok=True)

print('⬇️  Downloading dataset from Kaggle...')
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p {DATA_DIR} --unzip

print('✅ Dataset downloaded!')
!ls {DATA_DIR}

In [ ]:
import os

# Explore dataset structure
# The dataset has: New Plant Diseases Dataset/
#   train/ (38 folders)
#   valid/ (38 folders)
#   test/  (33 images)

# Find the train directory (path may vary after unzip)
for root, dirs, files in os.walk(DATA_DIR):
    if 'train' in dirs and 'valid' in dirs:
        DATASET_ROOT = root
        break

TRAIN_DIR = os.path.join(DATASET_ROOT, 'train')
VALID_DIR = os.path.join(DATASET_ROOT, 'valid')
TEST_DIR  = os.path.join(DATASET_ROOT, 'test')

classes = sorted(os.listdir(TRAIN_DIR))
NUM_CLASSES = len(classes)

print(f'📊 Dataset Summary:')
print(f'   Total classes  : {NUM_CLASSES}')
print(f'   Training images: {sum(len(os.listdir(os.path.join(TRAIN_DIR,c))) for c in classes):,}')
print(f'   Validation imgs: {sum(len(os.listdir(os.path.join(VALID_DIR,c))) for c in classes):,}')
print(f'\n🌿 Classes (first 10):')
for c in classes[:10]:
    print(f'   {c}')
print('   ...')

## Step 4 — Data Pipeline

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
import numpy as np
import matplotlib.pyplot as plt
import json
from tqdm.notebook import tqdm

# --- Config ---
IMG_SIZE    = 224       # EfficientNet-B3 default
BATCH_SIZE  = 32        # Adjust down to 16 if you get OOM errors
NUM_EPOCHS  = 15        # Enough for great accuracy with transfer learning
LR          = 3e-4      # Initial learning rate
NUM_WORKERS = 2
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'🖥️  Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ImageNet normalization stats (used since we're fine-tuning a pretrained model)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Training transforms — augmentation to improve generalization
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Validation transforms — no augmentation, just resize & normalize
val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Create datasets
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset   = datasets.ImageFolder(VALID_DIR, transform=val_transforms)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# Save class-to-index mapping (needed for inference in the webapp)
class_to_idx = train_dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

with open(f'{BASE_DIR}/class_indices.json', 'w') as f:
    json.dump(idx_to_class, f, indent=2)

print(f'✅ Data loaders ready!')
print(f'   Training batches  : {len(train_loader)}')
print(f'   Validation batches: {len(val_loader)}')
print(f'   Class mapping saved to Drive')

In [ ]:
# Visualize some training samples
def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    t = tensor.clone()
    for c, m, s in zip(range(3), mean, std):
        t[c] = t[c] * s + m
    return t.clamp(0, 1)

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Training Images', fontsize=14, fontweight='bold')
for i, ax in enumerate(axes.flat):
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    class_name = idx_to_class[labels[i].item()].replace('___', '\n').replace('_', ' ')
    ax.set_title(class_name, fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Step 5 — Model Definition (EfficientNet-B3)

In [ ]:
def build_model(num_classes, pretrained=True):
    """
    EfficientNet-B3 with a custom classification head.
    We freeze the backbone initially and only train the head,
    then unfreeze for full fine-tuning.
    """
    model = timm.create_model(
        'efficientnet_b3',
        pretrained=pretrained,
        num_classes=num_classes
    )
    return model

model = build_model(NUM_CLASSES)
model = model.to(DEVICE)

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'✅ EfficientNet-B3 loaded with pretrained ImageNet weights')
print(f'   Total parameters    : {total_params:,}')
print(f'   Trainable parameters: {trainable_params:,}')
print(f'   Output classes      : {NUM_CLASSES}')

## Step 6 — Training

In [ ]:
# Loss, optimizer, and learning rate scheduler
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # label smoothing helps generalization

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

# Cosine annealing scheduler — gradually reduces LR for better convergence
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)

# Mixed precision scaler (makes training 2-3x faster on GPU)
scaler = torch.cuda.amp.GradScaler()

print('✅ Optimizer: AdamW | Scheduler: Cosine Annealing | Mixed Precision: ON')

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc='Training', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():  # Mixed precision
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)

        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{correct/total:.3f}'})

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, desc='Validating', leave=False):
        images, labels = images.to(device), labels.to(device)
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)

    return total_loss / total, correct / total


print('✅ Training functions defined!')

In [ ]:
# =============================================
#  MAIN TRAINING LOOP
# =============================================

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
BEST_MODEL_PATH = f'{BASE_DIR}/models/best_model.pth'

print(f'🚀 Starting training for {NUM_EPOCHS} epochs...')
print(f'   Best model will be saved to: {BEST_MODEL_PATH}')
print('=' * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    current_lr = optimizer.param_groups[0]['lr']

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler, DEVICE)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, DEVICE)

    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    # Save best model
    saved = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'num_classes': NUM_CLASSES,
            'class_to_idx': class_to_idx,
        }, BEST_MODEL_PATH)
        saved = '  ⭐ BEST'

    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} | '
          f'LR: {current_lr:.2e} | '
          f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}{saved}')

print('=' * 65)
print(f'\n✅ Training complete! Best validation accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')

## Step 7 — Visualize Training Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_loss']) + 1)

# Loss curve
axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train Loss', markersize=4)
axes[0].plot(epochs, history['val_loss'],   'r-o', label='Val Loss',   markersize=4)
axes[0].set_title('Loss Curve', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(epochs, [a*100 for a in history['train_acc']], 'b-o', label='Train Acc', markersize=4)
axes[1].plot(epochs, [a*100 for a in history['val_acc']],   'r-o', label='Val Acc',   markersize=4)
axes[1].set_title('Accuracy Curve', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('EfficientNet-B3 — Plant Disease Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{BASE_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Training curves saved to Google Drive')

## Step 8 — Grad-CAM Visualization (Explainability)

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from PIL import Image

# Load best model
checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# The last conv block is the target layer for Grad-CAM
target_layer = [model.blocks[-1]]

cam = GradCAM(model=model, target_layers=target_layer)

# Get some validation images to visualize
val_dataset_no_aug = datasets.ImageFolder(VALID_DIR, transform=val_transforms)
sample_indices = np.random.choice(len(val_dataset_no_aug), 6, replace=False)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle('Grad-CAM: What the Model Looks At', fontsize=13, fontweight='bold')

for col, idx in enumerate(sample_indices):
    img_tensor, true_label = val_dataset_no_aug[idx]
    img_tensor_batch = img_tensor.unsqueeze(0).to(DEVICE)

    # Get prediction
    with torch.no_grad():
        output = model(img_tensor_batch)
        pred_label = output.argmax(1).item()
        confidence = torch.softmax(output, dim=1).max().item()

    # Generate Grad-CAM
    grayscale_cam = cam(input_tensor=img_tensor_batch)[0]

    # Denormalize for display
    raw_img = denormalize(img_tensor).permute(1, 2, 0).numpy()
    cam_img = show_cam_on_image(raw_img, grayscale_cam, use_rgb=True)

    # Original image
    axes[0, col].imshow(raw_img)
    axes[0, col].set_title(idx_to_class[true_label].replace('___', '\n').replace('_', ' '), fontsize=7)
    axes[0, col].axis('off')

    # Grad-CAM overlay
    correct = '✓' if pred_label == true_label else '✗'
    axes[1, col].imshow(cam_img)
    axes[1, col].set_title(f'{correct} {confidence*100:.1f}%', fontsize=9,
                           color='green' if pred_label == true_label else 'red')
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=10)
axes[1, 0].set_ylabel('Grad-CAM', fontsize=10)

plt.tight_layout()
plt.savefig(f'{BASE_DIR}/gradcam_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('🔥 Grad-CAM visualization saved!')

## Step 9 — Export Model for Deployment

In [ ]:
# Export as TorchScript (for production deployment)
model.eval()
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)

# TorchScript export
scripted_model = torch.jit.trace(model, dummy_input)
scripted_model.save(f'{BASE_DIR}/models/plant_disease_model_scripted.pt')
print('✅ TorchScript model saved!')

# Also save a clean checkpoint
torch.save({
    'model_state_dict': model.state_dict(),
    'num_classes': NUM_CLASSES,
    'class_to_idx': class_to_idx,
    'img_size': IMG_SIZE,
    'model_name': 'efficientnet_b3',
    'val_acc': best_val_acc,
}, f'{BASE_DIR}/models/plant_disease_final.pth')
print('✅ Final checkpoint saved!')

print(f'\n📦 All model files saved to: {BASE_DIR}/models/')
print(f'   - best_model.pth              (best during training)')
print(f'   - plant_disease_final.pth     (clean weights + metadata)')
print(f'   - plant_disease_model_scripted.pt  (TorchScript, ready for deploy)')
print(f'\n🏆 Final Val Accuracy: {best_val_acc*100:.2f}%')

## Step 10 — Quick Inference Test

In [ ]:
# Disease treatment database (top diseases)
DISEASE_INFO = {
    'healthy': {
        'description': 'The plant appears healthy with no signs of disease.',
        'treatment': 'Continue regular care — proper watering, fertilization, and sunlight.',
        'severity': 'None'
    },
    'Apple___Apple_scab': {
        'description': 'Fungal disease causing dark, scaly lesions on leaves and fruit.',
        'treatment': 'Apply fungicide (captan or myclobutanil). Remove and destroy infected leaves. Improve air circulation.',
        'severity': 'Moderate'
    },
    'Tomato___Early_blight': {
        'description': 'Fungal disease causing dark spots with concentric rings on lower leaves.',
        'treatment': 'Apply copper-based fungicide. Remove infected leaves. Avoid overhead watering.',
        'severity': 'Moderate'
    },
    'Tomato___Late_blight': {
        'description': 'Devastating disease causing dark water-soaked lesions. Can kill plant rapidly.',
        'treatment': 'Apply chlorothalonil or copper fungicide immediately. Remove infected tissue. High severity — act fast.',
        'severity': 'High'
    },
    # Add more as needed...
}


def predict_image(image_path, model, idx_to_class, device, top_k=3):
    """Run inference on a single image and return top-k predictions."""
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output = model(tensor)
        probs  = torch.softmax(output, dim=1)
        top_probs, top_indices = probs.topk(top_k, dim=1)

    results = []
    for prob, idx in zip(top_probs[0], top_indices[0]):
        class_name = idx_to_class[idx.item()]
        info = DISEASE_INFO.get(class_name, {
            'description': 'Please consult an agricultural expert.',
            'treatment': 'Apply general plant care practices.',
            'severity': 'Unknown'
        })
        results.append({
            'class': class_name,
            'confidence': prob.item(),
            **info
        })
    return results, img


# Test on a random validation image
test_class = np.random.choice(classes)
test_class_dir = os.path.join(VALID_DIR, test_class)
test_img_file  = np.random.choice(os.listdir(test_class_dir))
test_img_path  = os.path.join(test_class_dir, test_img_file)

results, original_img = predict_image(test_img_path, model, idx_to_class, DEVICE)

print(f'🌿 True label : {test_class}')
print(f'\n📊 Predictions:')
for i, r in enumerate(results, 1):
    print(f'   {i}. {r["class"]:50s} {r["confidence"]*100:.2f}%')
print(f'\n💊 Treatment for top prediction:')
print(f'   {results[0]["treatment"]}')

plt.figure(figsize=(4, 4))
plt.imshow(original_img)
plt.title(f'Predicted: {results[0]["class"]}\n({results[0]["confidence"]*100:.1f}%)', fontsize=9)
plt.axis('off')
plt.show()

## ✅ You're Done!

Your model is saved to Google Drive. Next steps:
1. Download the model files from Drive to your `D:/PlantDiseaseDetector/models/` folder
2. Build the FastAPI backend (inference server)
3. Build the React/Next.js frontend with camera upload
4. Deploy to Hugging Face Spaces or Render

**Files saved to Google Drive:**
- `models/best_model.pth` — best checkpoint
- `models/plant_disease_final.pth` — clean deployment weights
- `models/plant_disease_model_scripted.pt` — TorchScript (fastest inference)
- `class_indices.json` — class mapping for inference
- `training_curves.png` — loss & accuracy plots
- `gradcam_samples.png` — explainability visualizations